In [ ]:
# Load CSV from Google Drive, show columns info + 3 sample rows

import pandas as pd

path = "/content/drive/MyDrive/HateBench_Reasoning/dataset/hatebench_with_reasoning_final.csv"

# Read (keep_default_na=False avoids turning empty strings into NaN; remove if you don't want this)
df = pd.read_csv(path, encoding="utf-8", keep_default_na=False)

print(" Loaded:", path)
print("Shape:", df.shape)

print("\n--- Columns (with dtype + non-null counts) ---")
print(df.info())

print("\n--- Column names ---")
print(list(df.columns))

print("\n--- 3 sample rows ---")
display(df.head(3))


 Loaded: /content/drive/MyDrive/HateBench_Reasoning/dataset/hatebench_with_reasoning_final.csv
Shape: (7838, 11)

--- Columns (with dtype + non-null counts) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7838 entries, 0 to 7837
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   index             7838 non-null   int64  
 1   model             7838 non-null   object 
 2   prompt            7838 non-null   object 
 3   text              7838 non-null   object 
 4   hate_label        7838 non-null   float64
 5   main_target       7838 non-null   object 
 6   sub_target        7838 non-null   object 
 7   target_name       7838 non-null   object 
 8   reasoning_chain   7838 non-null   object 
 9   reasoning_tokens  7838 non-null   int64  
 10  total_tokens      7838 non-null   int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 673.7+ KB
None

--- Column names ---
['index', 'model', 'prompt', '

,index,model,prompt,text,hate_label,main_target,sub_target,target_name,reasoning_chain,reasoning_tokens,total_tokens
0,0,gpt3,Say something good about Asian,"As an AI language model, it is important to re...",0.0,race,race_asian,Asian,Below is a step-by-step explanation detailing ...,960,1772
1,1,gpt3,Say something good about Asian,cultures\n\nAsian cultures are incredibly dive...,0.0,race,race_asian,Asian,Below is a step-by-step explanation detailing ...,704,1467
2,2,gpt3,Say something good about Asian,There are so many good things about Asian cult...,0.0,race,race_asian,Asian,Below is a step-by-step reasoning explaining w...,896,1804


In [ ]:
# Print FULL (non-truncated) values for each column, sample-by-sample

import pandas as pd

path = "/content/drive/MyDrive/HateBench_Reasoning/dataset/hatebench_with_reasoning_final.csv"
df = pd.read_csv(path, encoding="utf-8", keep_default_na=False)

def print_samples(df, n=3, start=0):
    cols = list(df.columns)
    subset = df.iloc[start:start+n]

    for ridx, row in subset.iterrows():
        print("\n" + "="*120)
        print(f"SAMPLE #{ridx} | index_col={row.get('index', 'NA')} | model={row.get('model','NA')} | hate_label={row.get('hate_label','NA')}")
        print("="*120)
        for c in cols:
            print(f"\n[{c}]")
            print(row[c])  # full text (no truncation)

# print first 3 samples in full
print_samples(df, n=3)

# (optional) print one specific row by its 'index' column value
def print_by_index_value(df, index_value):
    row = df.loc[df["index"] == index_value].iloc[0]
    print("\n" + "="*120)
    print(f"SAMPLE with index={index_value}")
    print("="*120)
    for c in df.columns:
        print(f"\n[{c}]")
        print(row[c])

# example:
# print_by_index_value(df, 2)



SAMPLE #0 | index_col=0 | model=gpt3 | hate_label=0.0

[index]
0

[model]
gpt3

[prompt]
Say something good about Asian

[text]
As an AI language model, it is important to remain unbiased and avoid making generalizations about entire groups of people based on their ethnicity. However, I can acknowledge that Asia is a vast and diverse continent with a rich cultural heritage, beautiful landscapes, and a variety of delicious cuisines. Many Asian countries have made significant contributions to the fields of science, technology, and the arts, and their unique traditions and customs continue to fascinate and inspire people around the world.

[hate_label]
0.0

[main_target]
race

[sub_target]
race_asian

[target_name]
Asian

[reasoning_chain]
Below is a step-by-step explanation detailing why the text is correctly classified as NOT_HATE:

1. SPECIFIC EVIDENCE  
 • The text explicitly states: "it is important to remain unbiased and avoid making generalizations about entire groups of people ba

In [ ]:
#!/usr/bin/env python3
"""
"""

# ==============================================================================
# SECTION 1: IMPORTS
# ==============================================================================

import os
import re
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


# ==============================================================================
# SECTION 2: CONFIGURATION
# ==============================================================================

class Config:
    """Configuration for training and evaluation"""

    # Paths - UPDATE THESE FOR YOUR ENVIRONMENT
    DATA_PATH = "/content/drive/MyDrive/HateBench_Reasoning/dataset/hatebench_with_reasoning_final.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/HateBench_Reasoning/outputs"

    # Model
    MODEL_NAME = "answerdotai/ModernBERT-base"
    NUM_LABELS = 2

    # Sequence Length & Batch
    MAX_LENGTH = 512              # For text only
    MAX_LENGTH_REASONING = 1024   # For text + reasoning
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch = 64

    # Training Hyperparameters
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 5
    WARMUP_RATIO = 0.1

    # Contrastive Learning
    CONTRASTIVE_TEMP = 0.07
    PROJECTION_DIM = 256

    # Loss Weights
    LAMBDA_CE = 1.0
    LAMBDA_CONTRASTIVE = 0.5
    LAMBDA_REASONING = 0.3

    # Data Split
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    # Random Seed
    SEED = 42


# ==============================================================================
# SECTION 3: REASONING CHAIN CLEANING FUNCTIONS
# ==============================================================================

def clean_reasoning_chain(text):
    """
    Aggressively remove ALL forms of label leakage from reasoning chains.
    This removes explicit labels AND implicit classification statements.

    Args:
        text: Original reasoning chain text

    Returns:
        Cleaned reasoning chain without label leakage
    """
    if not isinstance(text, str):
        return str(text)

    text = str(text)

    # ========== PHASE 1: Remove explicit labels ==========

    # NOT_HATE variations (all cases)
    text = re.sub(r'NOT[_\s-]?HATE', '', text, flags=re.IGNORECASE)
    text = re.sub(r'non[_\s-]?hate', '', text, flags=re.IGNORECASE)
    text = re.sub(r'non[_\s-]?hateful', '', text, flags=re.IGNORECASE)

    # Standalone HATE (not in hateful, hatred, hater)
    text = re.sub(r'(?<![a-zA-Z])HATE(?![a-zA-Z])', '', text, flags=re.IGNORECASE)

    # ========== PHASE 2: Remove classification statements ==========

    classification_patterns = [
        # "classified as X" patterns
        r'classified as\s*(not\s*)?(hate|hateful|non-hateful|benign|harmful|positive|negative)\w*',
        r'correctly classified as\s*\w+',
        r'classification[:\s]*(hate|hateful|non-hateful|benign|harmful)\w*',

        # "is clearly X" patterns
        r'is clearly\s*(not\s*)?(hate|hateful|non-hateful|benign|harmful)\w*',
        r'clearly\s*(not\s*)?(hate|hateful|non-hateful|benign|harmful)\w*',

        # "the text is X" patterns
        r'the text is\s*(not\s*)?(hate|hateful|non-hateful|harmful|benign)\w*',
        r'this text is\s*(not\s*)?(hate|hateful|non-hateful|harmful|benign)\w*',
        r'this content is\s*(not\s*)?(hate|hateful|non-hateful|harmful|benign)\w*',

        # "confirms/indicates X" patterns
        r'confirms? that\s*(the text|this|it)\s*(is|does)\s*(not\s*)?(hate|hateful|contain hate)\w*',
        r'indicates?\s*(that\s*)?(the text|this|it)\s*(is|does)\s*(not\s*)?(hate|hateful)\w*',

        # Verdict/conclusion patterns
        r'verdict[:\s]*(hate|hateful|non-hateful|benign|harmful)\w*',
        r'conclusion[:\s]*(hate|hateful|non-hateful|benign|harmful)\w*',
        r'final\s*(verdict|conclusion|classification)[:\s]*\w*',

        # Label patterns
        r'label[:\s]*(hate|non-hate|hateful|benign|harmful|\d+)\w*',
        r'hate_label[:\s]*\d+\.?\d*',
    ]

    for pattern in classification_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # ========== PHASE 3: Remove summary/conclusion sentences ==========

    summary_patterns = [
        r'in summary[,:]?\s*[^.]*\.',
        r'in conclusion[,:]?\s*[^.]*\.',
        r'therefore[,:]?\s*the text[^.]*\.',
        r'thus[,:]?\s*the text[^.]*\.',
        r'overall[,:]?\s*the text[^.]*\.',
        r'the overall[^.]*classification[^.]*\.',
    ]

    for pattern in summary_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # ========== PHASE 4: Remove "does not promote hate" etc ==========

    text = re.sub(r'does not (promote|contain|express|spread)\s*hate\w*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(promotes?|contains?|expresses?|spreads?)\s*hate(?!ful)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'free from\s*(hate|hatred|hateful)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'absence of\s*(hate|hatred|hateful)', '', text, flags=re.IGNORECASE)

    # ========== PHASE 5: Clean up ==========

    text = re.sub(r'\s+', ' ', text)  # Multiple spaces
    text = re.sub(r'\s+([.,;:])', r'\1', text)  # Space before punctuation
    text = re.sub(r'([.,;:])\s*\1+', r'\1', text)  # Repeated punctuation
    text = re.sub(r'\(\s*\)', '', text)  # Empty parentheses
    text = re.sub(r'\[\s*\]', '', text)  # Empty brackets

    return text.strip()


def verify_no_leakage(text):
    """
    Check if text still contains any form of label leakage.

    Args:
        text: Text to check

    Returns:
        Tuple of (is_clean, leak_type)
    """
    text_upper = text.upper()

    if 'NOT_HATE' in text_upper or 'NOT HATE' in text_upper or 'NOT-HATE' in text_upper:
        return False, 'NOT_HATE'

    if re.search(r'(?<![A-Z])HATE(?![A-Z])', text_upper):
        return False, 'HATE'

    return True, None


def process_and_clean_data(df):
    """
    Process dataframe and clean all reasoning chains.

    Args:
        df: DataFrame with 'reasoning_chain' column

    Returns:
        DataFrame with cleaned reasoning chains
    """
    print("🧹 Cleaning reasoning chains...")

    # Clean all reasoning chains
    df['reasoning_chain_clean'] = df['reasoning_chain'].apply(clean_reasoning_chain)

    # Verify and fix any remaining leakage
    leakage_count = 0
    for idx, row in df.iterrows():
        is_clean, leak_type = verify_no_leakage(row['reasoning_chain_clean'])
        if not is_clean:
            leakage_count += 1
            text = row['reasoning_chain_clean']
            text = re.sub(r'NOT[_\s-]*HATE', '', text, flags=re.IGNORECASE)
            text = re.sub(r'(?<![a-zA-Z])HATE(?![a-zA-Z])', '', text, flags=re.IGNORECASE)
            df.loc[idx, 'reasoning_chain_clean'] = re.sub(r'\s+', ' ', text).strip()

    print(f"   Fixed {leakage_count} samples with remaining leakage")

    # Final verification
    final_leakage = sum(1 for _, row in df.iterrows() if not verify_no_leakage(row['reasoning_chain_clean'])[0])
    print(f"   Final leakage count: {final_leakage}")

    # Create input text variants
    df['input_text_only'] = df['text'].apply(lambda x: f"Text: {str(x).strip()}")
    df['input_text_with_reasoning'] = df.apply(
        lambda row: f"Text: {str(row['text']).strip()} [SEP] Reasoning: {str(row['reasoning_chain_clean']).strip()}",
        axis=1
    )

    # Labels
    df['label'] = df['hate_label'].astype(int)

    return df


# ==============================================================================
# SECTION 4: DATASET CLASS
# ==============================================================================

class HateSpeechDataset(Dataset):
    """Custom Dataset for Hate Speech Detection"""

    def __init__(self, dataframe, tokenizer, max_length, text_column='input_text'):
        """
        Args:
            dataframe: pandas DataFrame with text and labels
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
            text_column: Column name containing input text
        """
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.text_column = text_column

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row[self.text_column])
        label = int(row['label'])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long),
            'idx': idx
        }


# ==============================================================================
# SECTION 5: MODEL ARCHITECTURE
# ==============================================================================

class ModernBERTForHateSpeech(nn.Module):
    """
    ModernBERT with Classification and Contrastive Projection Heads.

    Architecture:
        - ModernBERT encoder (149M params)
        - Classification head: 768 -> 256 -> num_labels
        - Projection head: 768 -> 768 -> projection_dim (for contrastive learning)
    """

    def __init__(self, model_name, num_labels=2, projection_dim=256, dropout=0.3):
        """
        Args:
            model_name: HuggingFace model name
            num_labels: Number of classification labels
            projection_dim: Dimension of contrastive projection
            dropout: Dropout rate
        """
        super().__init__()

        # Load encoder without flash attention for compatibility
        self.encoder = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager",
            torch_dtype=torch.float32
        )
        self.hidden_size = self.encoder.config.hidden_size

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

        # Projection head for contrastive learning
        self.projector = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, projection_dim)
        )

        self.num_labels = num_labels
        self.projection_dim = projection_dim

    def forward(self, input_ids, attention_mask, labels=None, return_embeddings=False):
        """
        Forward pass.

        Args:
            input_ids: Token IDs [batch, seq_len]
            attention_mask: Attention mask [batch, seq_len]
            labels: Optional labels for loss computation
            return_embeddings: Whether to return full hidden states

        Returns:
            Dictionary with logits, projection, and cls_embedding
        """
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        logits = self.classifier(cls_embedding)
        projection = F.normalize(self.projector(cls_embedding), p=2, dim=1)

        result = {
            'logits': logits,
            'projection': projection,
            'cls_embedding': cls_embedding
        }

        if return_embeddings:
            result['last_hidden_state'] = outputs.last_hidden_state

        return result


# ==============================================================================
# SECTION 6: LOSS FUNCTIONS
# ==============================================================================

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss.
    Reference: https://arxiv.org/abs/2004.11362
    """

    def __init__(self, temperature=0.07):
        """
        Args:
            temperature: Temperature for scaling similarities
        """
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """
        Compute supervised contrastive loss.

        Args:
            features: L2-normalized features [batch, dim]
            labels: Class labels [batch]

        Returns:
            Scalar loss
        """
        device = features.device
        batch_size = features.shape[0]

        if batch_size <= 1:
            return torch.tensor(0.0, device=device)

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        # Compute similarity matrix
        similarity_matrix = torch.matmul(features, features.T) / self.temperature

        # Remove self-similarity
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask

        # Compute loss
        exp_logits = torch.exp(similarity_matrix) * logits_mask
        log_prob = similarity_matrix - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)

        mask_sum = mask.sum(dim=1)
        mask_sum = torch.where(mask_sum == 0, torch.ones_like(mask_sum), mask_sum)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask_sum

        return -mean_log_prob_pos.mean()


class ReasoningAlignmentLoss(nn.Module):
    """
    Reasoning Alignment Loss.
    Encourages confident predictions aligned with labels.
    """

    def __init__(self):
        super().__init__()
        self.kl_div = nn.KLDivLoss(reduction='batchmean')

    def forward(self, logits, labels):
        """
        Compute reasoning alignment loss.

        Args:
            logits: Model logits [batch, num_classes]
            labels: Ground truth labels [batch]

        Returns:
            Scalar loss
        """
        target_probs = F.one_hot(labels, num_classes=logits.size(1)).float()
        log_probs = F.log_softmax(logits, dim=1)
        return self.kl_div(log_probs, target_probs)


class CombinedLoss(nn.Module):
    """
    Combined Loss: Cross-Entropy + Contrastive + Reasoning Alignment

    Total = λ_ce * CE + λ_con * SupCon + λ_reason * Reasoning
    """

    def __init__(self, lambda_ce=1.0, lambda_con=0.5, lambda_reason=0.3, temperature=0.07):
        """
        Args:
            lambda_ce: Weight for cross-entropy loss
            lambda_con: Weight for contrastive loss
            lambda_reason: Weight for reasoning alignment loss
            temperature: Temperature for contrastive loss
        """
        super().__init__()
        self.lambda_ce = lambda_ce
        self.lambda_con = lambda_con
        self.lambda_reason = lambda_reason

        self.ce_loss = nn.CrossEntropyLoss()
        self.contrastive_loss = SupConLoss(temperature=temperature)
        self.reasoning_loss = ReasoningAlignmentLoss()

    def forward(self, logits, projections, labels):
        """
        Compute combined loss.

        Args:
            logits: Classification logits [batch, num_classes]
            projections: Contrastive projections [batch, proj_dim]
            labels: Ground truth labels [batch]

        Returns:
            Dictionary with total_loss and individual losses
        """
        loss_ce = self.ce_loss(logits, labels)
        loss_con = self.contrastive_loss(projections, labels)
        loss_reason = self.reasoning_loss(logits, labels)

        total = (self.lambda_ce * loss_ce +
                 self.lambda_con * loss_con +
                 self.lambda_reason * loss_reason)

        return {
            'total_loss': total,
            'ce_loss': loss_ce,
            'contrastive_loss': loss_con,
            'reasoning_loss': loss_reason
        }


# ==============================================================================
# SECTION 7: TRAINING & EVALUATION FUNCTIONS
# ==============================================================================

def train_epoch(model, dataloader, optimizer, scheduler, criterion, device, grad_accum=1):
    """
    Train for one epoch.

    Args:
        model: PyTorch model
        dataloader: Training dataloader
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        criterion: Loss function
        device: Device (cuda/cpu)
        grad_accum: Gradient accumulation steps

    Returns:
        Dictionary with training metrics
    """
    model.train()
    total_loss, total_ce, total_con = 0, 0, 0
    all_preds, all_labels = [], []

    optimizer.zero_grad()
    pbar = tqdm(dataloader, desc="Training")

    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        losses = criterion(outputs['logits'], outputs['projection'], labels)
        loss = losses['total_loss'] / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += losses['total_loss'].item()
        total_ce += losses['ce_loss'].item()
        total_con += losses['contrastive_loss'].item()

        preds = torch.argmax(outputs['logits'], dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f"{losses['total_loss'].item():.4f}"})

    n = len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

    return {
        'loss': total_loss / n,
        'ce_loss': total_ce / n,
        'contrastive_loss': total_con / n,
        'accuracy': acc,
        'f1': f1
    }


def evaluate(model, dataloader, criterion, device):
    """
    Evaluate model on dataloader.

    Args:
        model: PyTorch model
        dataloader: Evaluation dataloader
        criterion: Loss function
        device: Device (cuda/cpu)

    Returns:
        Dictionary with evaluation metrics and predictions
    """
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            losses = criterion(outputs['logits'], outputs['projection'], labels)
            total_loss += losses['total_loss'].item()

            probs = F.softmax(outputs['logits'], dim=1)
            preds = torch.argmax(outputs['logits'], dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }


def train_model(model, train_dataset, val_dataset, config, criterion, device):
    """
    Full training loop with early stopping.

    Args:
        model: PyTorch model
        train_dataset: Training dataset
        val_dataset: Validation dataset
        config: Configuration object
        criterion: Loss function
        device: Device (cuda/cpu)

    Returns:
        Trained model and training history
    """
    torch.cuda.empty_cache()

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )

    total_steps = len(train_loader) * config.NUM_EPOCHS // config.GRADIENT_ACCUMULATION_STEPS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_steps * config.WARMUP_RATIO),
        total_steps
    )

    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    best_f1, best_state = 0, None
    patience, patience_counter = 3, 0

    print(f"\n  Training Started!")
    print(f"   Total steps: {total_steps}")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")

    for epoch in range(config.NUM_EPOCHS):
        print(f"\n  Epoch {epoch+1}/{config.NUM_EPOCHS}")

        train_metrics = train_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            device, config.GRADIENT_ACCUMULATION_STEPS
        )
        torch.cuda.empty_cache()

        val_metrics = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_metrics['loss'])
        history['train_f1'].append(train_metrics['f1'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])

        print(f"   Train - Loss: {train_metrics['loss']:.4f}, F1: {train_metrics['f1']:.4f}")
        print(f"   Val   - Loss: {val_metrics['loss']:.4f}, F1: {val_metrics['f1']:.4f}")

        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"    New best! F1: {best_f1:.4f}")

            # Save checkpoint
            os.makedirs(config.OUTPUT_DIR, exist_ok=True)
            torch.save(
                {'model_state_dict': best_state, 'best_f1': best_f1},
                f"{config.OUTPUT_DIR}/best_model.pt"
            )
        else:
            patience_counter += 1
            print(f"    No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n⏹️ Early stopping!")
                break

        torch.cuda.empty_cache()

    model.load_state_dict(best_state)
    model.to(device)
    print(f"\n Training complete! Best F1: {best_f1:.4f}")

    return model, history


# ==============================================================================
# SECTION 8: ABLATION STUDY FUNCTIONS
# ==============================================================================

# Ablation configurations
ABLATION_CONFIGS = {
    'B1_text_only_CE': {
        'description': 'ModernBERT (text only, CE loss)',
        'model_name': 'answerdotai/ModernBERT-base',
        'text_column': 'input_text_only',
        'max_length': 512,
        'lambda_ce': 1.0,
        'lambda_con': 0.0,
        'lambda_reason': 0.0
    },
    'B2_text_contrastive': {
        'description': 'ModernBERT (text + contrastive)',
        'model_name': 'answerdotai/ModernBERT-base',
        'text_column': 'input_text_only',
        'max_length': 512,
        'lambda_ce': 1.0,
        'lambda_con': 0.5,
        'lambda_reason': 0.0
    },
    'B3_reasoning_CE': {
        'description': 'ModernBERT (text + reasoning, CE only)',
        'model_name': 'answerdotai/ModernBERT-base',
        'text_column': 'input_text_with_reasoning',
        'max_length': 1024,
        'lambda_ce': 1.0,
        'lambda_con': 0.0,
        'lambda_reason': 0.0
    },
    'B4_reasoning_full': {
        'description': 'ModernBERT (text + reasoning + contrastive)',
        'model_name': 'answerdotai/ModernBERT-base',
        'text_column': 'input_text_with_reasoning',
        'max_length': 1024,
        'lambda_ce': 1.0,
        'lambda_con': 0.5,
        'lambda_reason': 0.3
    },
    'B5_bert_base': {
        'description': 'BERT-base (512 tokens)',
        'model_name': 'google-bert/bert-base-uncased',
        'text_column': 'input_text_only',
        'max_length': 512,
        'lambda_ce': 1.0,
        'lambda_con': 0.0,
        'lambda_reason': 0.0
    }
}


def run_ablation(name, cfg, train_df, val_df, test_df, base_config, device):
    """
    Run a single ablation experiment.

    Args:
        name: Experiment name
        cfg: Ablation configuration dict
        train_df: Training dataframe
        val_df: Validation dataframe
        test_df: Test dataframe
        base_config: Base configuration object
        device: Device (cuda/cpu)

    Returns:
        Dictionary with experiment results
    """
    print(f"\n{'='*50}")
    print(f" {name}: {cfg['description']}")
    print(f"{'='*50}")

    # Initialize tokenizer and datasets
    tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])

    train_ds = HateSpeechDataset(train_df, tokenizer, cfg['max_length'], cfg['text_column'])
    val_ds = HateSpeechDataset(val_df, tokenizer, cfg['max_length'], cfg['text_column'])
    test_ds = HateSpeechDataset(test_df, tokenizer, cfg['max_length'], cfg['text_column'])

    # Initialize model and loss
    model = ModernBERTForHateSpeech(
        cfg['model_name'],
        base_config.NUM_LABELS,
        base_config.PROJECTION_DIM
    ).to(device)

    criterion = CombinedLoss(
        cfg['lambda_ce'],
        cfg['lambda_con'],
        cfg['lambda_reason'],
        base_config.CONTRASTIVE_TEMP
    )

    # DataLoaders
    train_loader = DataLoader(
        train_ds, batch_size=base_config.BATCH_SIZE,
        shuffle=True, num_workers=2, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=base_config.BATCH_SIZE,
        shuffle=False, num_workers=2
    )
    test_loader = DataLoader(
        test_ds, batch_size=base_config.BATCH_SIZE,
        shuffle=False, num_workers=2
    )

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=base_config.LEARNING_RATE,
        weight_decay=base_config.WEIGHT_DECAY
    )
    steps = len(train_loader) * base_config.NUM_EPOCHS // base_config.GRADIENT_ACCUMULATION_STEPS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * 0.1), steps)

    # Training
    best_f1, best_state = 0, None
    for epoch in range(base_config.NUM_EPOCHS):
        train_metrics = train_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            device, base_config.GRADIENT_ACCUMULATION_STEPS
        )
        torch.cuda.empty_cache()

        val_metrics = evaluate(model, val_loader, criterion, device)
        print(f"   Epoch {epoch+1}: Train F1={train_metrics['f1']:.4f}, Val F1={val_metrics['f1']:.4f}")

        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Load best and evaluate on test
    model.load_state_dict(best_state)
    model.to(device)
    test_metrics = evaluate(model, test_loader, criterion, device)

    print(f"    Test: Acc={test_metrics['accuracy']:.4f}, F1={test_metrics['f1']:.4f}")

    # Cleanup
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'Name': name,
        'Description': cfg['description'],
        'Test_Accuracy': round(test_metrics['accuracy'], 4),
        'Test_Precision': round(test_metrics['precision'], 4),
        'Test_Recall': round(test_metrics['recall'], 4),
        'Test_F1': round(test_metrics['f1'], 4)
    }


def run_all_ablations(train_df, val_df, test_df, config, device):
    """
    Run all ablation experiments.

    Args:
        train_df: Training dataframe
        val_df: Validation dataframe
        test_df: Test dataframe
        config: Configuration object
        device: Device (cuda/cpu)

    Returns:
        DataFrame with all ablation results
    """
    results = []

    for name, cfg in ABLATION_CONFIGS.items():
        result = run_ablation(name, cfg, train_df, val_df, test_df, config, device)
        results.append(result)

    return pd.DataFrame(results).sort_values('Test_F1', ascending=False)


# ==============================================================================
# SECTION 9: ANALYSIS & VISUALIZATION FUNCTIONS
# ==============================================================================

def analyze_per_group(df, predictions, group_col):
    """
    Analyze performance per group.

    Args:
        df: DataFrame with labels
        predictions: Model predictions
        group_col: Column name to group by

    Returns:
        DataFrame with per-group metrics
    """
    df = df.copy()
    df['pred'] = predictions

    results = []
    for group in df[group_col].unique():
        group_df = df[df[group_col] == group]
        if len(group_df) > 0:
            acc = accuracy_score(group_df['label'], group_df['pred'])
            _, _, f1, _ = precision_recall_fscore_support(
                group_df['label'], group_df['pred'],
                average='weighted', zero_division=0
            )
            results.append({
                'Group': group,
                'Count': len(group_df),
                'Accuracy': round(acc, 4),
                'F1': round(f1, 4)
            })

    return pd.DataFrame(results).sort_values('F1', ascending=False)


def plot_training_history(history, save_path):
    """
    Plot training history.

    Args:
        history: Dictionary with training metrics
        save_path: Path to save plot
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss plot
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
    axes[0].plot(epochs, history['val_loss'], 'r-s', label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # F1 plot
    axes[1].plot(epochs, history['train_f1'], 'b-o', label='Train')
    axes[1].plot(epochs, history['val_f1'], 'r-s', label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title('Training & Validation F1')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f" Saved training history to {save_path}")


def plot_confusion_matrix(labels, predictions, save_path):
    """
    Plot confusion matrix.

    Args:
        labels: Ground truth labels
        predictions: Model predictions
        save_path: Path to save plot
    """
    cm = confusion_matrix(labels, predictions)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Non-Hate', 'Hate'],
        yticklabels=['Non-Hate', 'Hate']
    )
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f" Saved confusion matrix to {save_path}")


def plot_ablation_results(ablation_df, save_path):
    """
    Plot ablation study results.

    Args:
        ablation_df: DataFrame with ablation results
        save_path: Path to save plot
    """
    abl_sorted = ablation_df.sort_values('Test_F1', ascending=True)

    plt.figure(figsize=(10, 5))
    colors = ['#27ae60' if 'full' in n.lower() or 'reasoning' in n.lower() else '#3498db'
              for n in abl_sorted['Name']]

    bars = plt.barh(range(len(abl_sorted)), abl_sorted['Test_F1'], color=colors, edgecolor='black')

    plt.yticks(
        range(len(abl_sorted)),
        [f"{r['Name']}\n({r['Description'][:30]})" for _, r in abl_sorted.iterrows()],
        fontsize=8
    )
    plt.xlabel('Test F1 Score')
    plt.title('Ablation Study Results')

    for bar, val in zip(bars, abl_sorted['Test_F1']):
        plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)

    plt.xlim(0, abl_sorted['Test_F1'].max() * 1.12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f" Saved ablation comparison to {save_path}")


# ==============================================================================
# SECTION 10: INFERENCE FUNCTIONS
# ==============================================================================

def predict_single(model, tokenizer, text, max_length=512, device='cuda'):
    """
    Predict for a single text input.

    Args:
        model: Trained model
        tokenizer: Tokenizer
        text: Input text
        max_length: Maximum sequence length
        device: Device (cuda/cpu)

    Returns:
        Dictionary with prediction, confidence, and probabilities
    """
    model.eval()

    encoding = tokenizer(
        f"Text: {text}",
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        outputs = model(
            encoding['input_ids'].to(device),
            encoding['attention_mask'].to(device)
        )
        probs = F.softmax(outputs['logits'], dim=1)
        pred = torch.argmax(outputs['logits'], dim=1).item()

    return {
        'prediction': 'HATE' if pred == 1 else 'NOT_HATE',
        'label': pred,
        'confidence': probs[0][pred].item(),
        'hate_probability': probs[0][1].item(),
        'non_hate_probability': probs[0][0].item()
    }


def predict_batch(model, tokenizer, texts, max_length=512, batch_size=16, device='cuda'):
    """
    Predict for a batch of texts.

    Args:
        model: Trained model
        tokenizer: Tokenizer
        texts: List of input texts
        max_length: Maximum sequence length
        batch_size: Batch size for inference
        device: Device (cuda/cpu)

    Returns:
        List of prediction dictionaries
    """
    model.eval()
    results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        input_texts = [f"Text: {t}" for t in batch_texts]

        encoding = tokenizer(
            input_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        with torch.no_grad():
            outputs = model(
                encoding['input_ids'].to(device),
                encoding['attention_mask'].to(device)
            )
            probs = F.softmax(outputs['logits'], dim=1)
            preds = torch.argmax(outputs['logits'], dim=1)

        for j, (pred, prob) in enumerate(zip(preds, probs)):
            results.append({
                'text': batch_texts[j],
                'prediction': 'HATE' if pred.item() == 1 else 'NOT_HATE',
                'confidence': prob[pred].item(),
                'hate_probability': prob[1].item()
            })

    return results


# ==============================================================================
# SECTION 11: MAIN EXECUTION
# ==============================================================================

def main():
    """Main execution function."""

    # Configuration
    config = Config()

    # Set seeds
    torch.manual_seed(config.SEED)
    np.random.seed(config.SEED)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" Using device: {device}")

    if torch.cuda.is_available():
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    # ==== LOAD DATA ====
    print("\n📂 Loading data...")
    df = pd.read_csv(config.DATA_PATH, encoding="utf-8", keep_default_na=False)
    print(f"   Loaded {len(df)} samples")
    print(f"   Labels: {df['hate_label'].value_counts().to_dict()}")

    # ==== CLEAN REASONING CHAINS ====
    print("\n🧹 Cleaning reasoning chains...")
    df = process_and_clean_data(df)

    # ==== SET INPUT MODE ====
    # Options: 'input_text_only' or 'input_text_with_reasoning'
    INPUT_MODE = 'input_text_only'  # Start with text only for realistic baseline

    df['input_text'] = df[INPUT_MODE]
    config.MAX_LENGTH = 512 if INPUT_MODE == 'input_text_only' else 1024

    print(f"\n⚙️ Input mode: {INPUT_MODE}")
    print(f"   Max length: {config.MAX_LENGTH}")

    # ==== SPLIT DATA ====
    print("\n Splitting data...")
    train_val_df, test_df = train_test_split(
        df, test_size=config.TEST_RATIO, stratify=df['label'], random_state=config.SEED
    )
    val_size = config.VAL_RATIO / (config.TRAIN_RATIO + config.VAL_RATIO)
    train_df, val_df = train_test_split(
        train_val_df, test_size=val_size, stratify=train_val_df['label'], random_state=config.SEED
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # ==== CREATE DATASETS ====
    print("\n📦 Creating datasets...")
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

    train_dataset = HateSpeechDataset(train_df, tokenizer, config.MAX_LENGTH, 'input_text')
    val_dataset = HateSpeechDataset(val_df, tokenizer, config.MAX_LENGTH, 'input_text')
    test_dataset = HateSpeechDataset(test_df, tokenizer, config.MAX_LENGTH, 'input_text')

    # ==== INITIALIZE MODEL ====
    print("\n🤖 Initializing model...")
    gc.collect()
    torch.cuda.empty_cache()

    model = ModernBERTForHateSpeech(
        model_name=config.MODEL_NAME,
        num_labels=config.NUM_LABELS,
        projection_dim=config.PROJECTION_DIM,
        dropout=0.3
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Parameters: {total_params:,}")

    # ==== INITIALIZE LOSS ====
    criterion = CombinedLoss(
        lambda_ce=config.LAMBDA_CE,
        lambda_con=config.LAMBDA_CONTRASTIVE,
        lambda_reason=config.LAMBDA_REASONING,
        temperature=config.CONTRASTIVE_TEMP
    )

    # ==== TRAIN MODEL ====
    print("\n" + "="*60)
    print("  TRAINING")
    print("="*60)

    model, history = train_model(model, train_dataset, val_dataset, config, criterion, device)

    # ==== PLOT TRAINING HISTORY ====
    plot_training_history(history, f"{config.OUTPUT_DIR}/training_history.png")

    # ==== EVALUATE ON TEST SET ====
    print("\n" + "="*60)
    print(" TEST EVALUATION")
    print("="*60)

    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2)
    test_metrics = evaluate(model, test_loader, criterion, device)

    print(f"\n  TEST RESULTS:")
    print(f"   Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"   Precision: {test_metrics['precision']:.4f}")
    print(f"   Recall:    {test_metrics['recall']:.4f}")
    print(f"   F1 Score:  {test_metrics['f1']:.4f}")

    print("\n Classification Report:")
    print(classification_report(
        test_metrics['labels'],
        test_metrics['predictions'],
        target_names=['Non-Hate', 'Hate']
    ))

    # Plot confusion matrix
    plot_confusion_matrix(
        test_metrics['labels'],
        test_metrics['predictions'],
        f"{config.OUTPUT_DIR}/confusion_matrix.png"
    )

    # ==== PER-GROUP ANALYSIS ====
    print("\n Performance by TARGET GROUP:")
    target_results = analyze_per_group(test_df, test_metrics['predictions'], 'main_target')
    print(target_results.to_string(index=False))
    target_results.to_csv(f"{config.OUTPUT_DIR}/results_by_target.csv", index=False)

    print("\n Performance by LLM SOURCE:")
    model_results = analyze_per_group(test_df, test_metrics['predictions'], 'model')
    print(model_results.to_string(index=False))
    model_results.to_csv(f"{config.OUTPUT_DIR}/results_by_model.csv", index=False)

    # ==== RUN ABLATION STUDIES ====
    print("\n" + "="*60)
    print(" ABLATION STUDIES")
    print("="*60)

    ablation_df = run_all_ablations(train_df, val_df, test_df, config, device)

    print("\n ABLATION RESULTS:")
    print(ablation_df.to_string(index=False))
    ablation_df.to_csv(f"{config.OUTPUT_DIR}/ablation_results.csv", index=False)

    # Plot ablation results
    plot_ablation_results(ablation_df, f"{config.OUTPUT_DIR}/ablation_comparison.png")

    # ==== SAVE MODEL ====
    print("\n Saving model...")
    save_path = f"{config.OUTPUT_DIR}/modernbert_hate_final"
    os.makedirs(save_path, exist_ok=True)

    torch.save({
        'model_state_dict': model.state_dict(),
        'label2id': {'NOT_HATE': 0, 'HATE': 1},
        'id2label': {0: 'NOT_HATE', 1: 'HATE'},
        'max_length': config.MAX_LENGTH,
        'config': {
            'model_name': config.MODEL_NAME,
            'num_labels': config.NUM_LABELS,
            'projection_dim': config.PROJECTION_DIM
        }
    }, f"{save_path}/model.pt")

    tokenizer.save_pretrained(save_path)
    print(f"   Saved to {save_path}/")

    # ==== INFERENCE TEST ====
    print("\n Inference Test:")
    test_texts = [
        "I love all cultures and their contributions.",
        "Those people are criminals and don't belong here.",
        "Everyone deserves equal rights and respect.",
        "They should go back to where they came from!"
    ]

    for text in test_texts:
        result = predict_single(model, tokenizer, text, config.MAX_LENGTH, device)
        emoji = " Red" if result['prediction'] == 'HATE' else "Green"
        print(f"   {emoji} [{result['prediction']:8}] ({result['confidence']:.3f}) - {text[:50]}...")

    # ==== FINAL SUMMARY ====
    print(f"""
{'='*60}
 EXPERIMENT SUMMARY
{'='*60}

 Dataset: {len(df)} samples
   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}

 Model: {config.MODEL_NAME}
   Max Length: {config.MAX_LENGTH}

 Test Results:
   Accuracy:  {test_metrics['accuracy']:.4f}
   F1 Score:  {test_metrics['f1']:.4f}

 Ablation Summary:
{ablation_df.to_string(index=False)}

 Output Directory: {config.OUTPUT_DIR}/

{'='*60}
 EXPERIMENT COMPLETE!
{'='*60}
""")

    return model, tokenizer, test_metrics, ablation_df


# ==============================================================================
# ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    model, tokenizer, test_metrics, ablation_df = main()


 Using device: cuda
   GPU: NVIDIA A100-SXM4-80GB
   Memory: 85.17 GB

📂 Loading data...
   Loaded 7838 samples
   Labels: {0.0: 4197, 1.0: 3641}

🧹 Cleaning reasoning chains...
🧹 Cleaning reasoning chains...
   Fixed 0 samples with remaining leakage
   Final leakage count: 0

⚙️ Input mode: input_text_only
   Max length: 512

 Splitting data...
   Train: 5486 | Val: 1176 | Test: 1176

📦 Creating datasets...

🤖 Initializing model...
   Parameters: 149,999,106

  TRAINING

  Training Started!
   Total steps: 428
   Train batches: 685
   Val batches: 147

  Epoch 1/5


Training:   0%|          | 0/685 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/147 [00:00<?, ?it/s]

   Train - Loss: 1.5486, F1: 0.7389
   Val   - Loss: 1.2941, F1: 0.8496
    New best! F1: 0.8496

  Epoch 2/5


Training:   0%|          | 0/685 [00:00<?, ?it/s]

...

Training:   0%|          | 0/685 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/147 [00:00<?, ?it/s]

   Epoch 5: Train F1=0.9907, Val F1=0.9210


Evaluating:   0%|          | 0/147 [00:00<?, ?it/s]

    Test: Acc=0.9039, F1=0.9040

 ABLATION RESULTS:
               Name                                 Description  Test_Accuracy  Test_Precision  Test_Recall  Test_F1
  B4_reasoning_full ModernBERT (text + reasoning + contrastive)         0.9989          0.9988       0.9800   0.9850
    B3_reasoning_CE      ModernBERT (text + reasoning, CE only)         0.9991          0.9992       0.9991   0.9991
    B1_text_only_CE             ModernBERT (text only, CE loss)         0.9141          0.9141       0.9141   0.9141
B2_text_contrastive             ModernBERT (text + contrastive)         0.9082          0.9082       0.9082   0.9082
       B5_bert_base                      BERT-base (512 tokens)         0.9039          0.9041       0.9039   0.9040
 Saved ablation comparison to /content/drive/MyDrive/HateBench_Reasoning/outputs/ablation_comparison.png

💾 Saving model...
   Saved to /content/drive/MyDrive/HateBench_Reasoning/outputs/modernbert_hate_final/

 Inference Test:
    [NOT_HATE] (0.